# Plot GEN files
Plot one or more GEN files side by side, e.g. to compare GENPUZZLE input and output.

In [ ]:
from pathlib import Path

import imod
import matplotlib.pyplot as plt

from genpuzzle import genpuzzle
from logging_helpers import init_logging

init_logging()

In [ ]:
data_dir = Path("tests/data")
genfile_in = data_dir / "genpuzzle_test.gen"
tolerances = [0.0, 5.0]

genfiles = [genfile_in]
for tolerance in tolerances:
    genfile_out = data_dir / f"out_tol{tolerance:g}.gen"
    genpuzzle(genfile_in, genfile_out, tolerance)
    genfiles.append(genfile_out)

In [ ]:
fig, axes = plt.subplots(
    1, len(genfiles), figsize=(5 * len(genfiles), 5), sharex=True, sharey=True, squeeze=False
)
for ax, genfile in zip(axes[0], genfiles):
    gdf = imod.formats.gen.read(genfile)
    gdf.plot(ax=ax, column="id", cmap="tab10", linewidth=2)
    for line_id, line in zip(gdf["id"], gdf.geometry):
        (x0, y0), (x1, y1) = line.coords[0], line.coords[-1]
        ax.plot([x0, x1], [y0, y1], "ko", markersize=4)
        mid = line.interpolate(0.5, normalized=True)
        ax.annotate(str(line_id), (mid.x, mid.y), fontsize=11, weight="bold")
    ax.set_title(f"{genfile.name} ({len(gdf)} lines)")
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
fig.tight_layout()